<a href="https://colab.research.google.com/github/trainocate-japan/developing-agentic-ai-with-langchain/blob/main/chap04/hands-on/chap04_handson_4A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ハンズオン 4-A: Checkpointer と LangSmith トレース

**研修コース「LangChain による Agentic AI 開発実践」/ 第4章「メモリと可観測性」**

このハンズオンは、講師の解説を聞きながら**作成済みのセルを上から順に一緒に実行する**形式です。
受講者がコードを書く場面はありません (コードを書くのは演習 4-B です)。
まずは「動かして観察する」ことに集中してください。

題材は中立な教材として**天気エージェント** (`get_weather`) を使います
(ヘルプデスクへの応用は演習 4-B で行います)。第3章で作った天気エージェントに、
**会話の記憶 (checkpointer)** と **トレース (LangSmith)** を加えるのがゴールです。

## この Notebook で学ぶこと

第3章の最後に、エージェントが会話を覚えていないこと (ステートレス性) を体験しました。
本章では、それを **2 行の追加**で解決し、さらにエージェントの内部を**コード変更ゼロ**で可視化します。

1. **checkpointer を追加する (4-2)** — `InMemorySaver` を import し `create_agent(..., checkpointer=...)`。
   checkpointer **なし**版で「忘れる」→ **あり**版で「覚える」の対比を体験する
2. **thread_id で継続・分離する (4-2)** — `config={"configurable": {"thread_id": "1"}}` で会話を継続。
   `thread_id` を変えると記憶が分離され、元に戻すと記憶が残ることを確認する。
   `result["messages"]` の件数が invoke ごとに累積することも観察する
3. **LangSmith でトレースを有効化する (4-4)** — 環境変数 2 つ + プロジェクト名で、
   コード変更なしにトレースが記録される。`config` に `tags` / `metadata` を付ける例も試す
4. **トレース読解ワークシート (4-4)** — smith.langchain.com でトレースを開き、
   「3 点チェック (ループ周回数 / ツールと引数 / トークン消費)」を記入する

## 前提条件

- **Google アカウント**を持っていること
- このファイルを **Google Colab** で開いていること
- Colab の **[シークレット]** (左サイドバーの鍵アイコン 🔑) に **2 種類のキー**が登録されていること
  - **`OPENAI_API_KEY`** — 第1章の演習 1-1 で登録済みのはずです。
    未登録でも、後述の「0-2. OpenAI API キーのセットアップ」で登録できます
  - **`LANGSMITH_API_KEY`** — 本章で**新しく必要**になります。
    後述の「0-3. LangSmith のセットアップ」の手順で、無料アカウント作成 → API キー発行 → シークレット登録を行います
- インターネット接続 (API を呼び出します)

## 所要時間

約 30 分 (4-2・4-4 を、講師の解説を含めて一気通貫で実行)

---
> **モデル名について**: 本教材ではモデル名を変数 `MODEL` に集約しています。教材中の例は `MODEL = "openai:gpt-5.4"` ですが、
> **研修実施時には講師が指定する最新モデル名に差し替えてください**。1 箇所 (準備セル) を直すだけで全セルに反映されます。

## 0. セットアップ

### 0-1. 依存パッケージのインストール

LangChain v1 本体 (`langchain`) と OpenAI 統合 (`langchain-openai`) をインストールします。
checkpointer (`InMemorySaver`) は `langchain` が依存する `langgraph` に同梱されているため、
追加のインストールは不要です。LangSmith のトレース送信機能も追加パッケージなしで動きます。

> 研修実施時は再現性のため、バージョンをピン留めすることを推奨します
> (本コースの基盤は **langchain 1.3.x / langchain-openai 1.3.x** です)。
> 例: `!pip install -U "langchain==1.3.7" "langchain-openai==1.3.0"`

In [ ]:
# LangChain v1 本体と OpenAI 統合を最新版へインストール/更新
# 研修実施時はバージョンをピン留め推奨 (langchain 1.3.x / langchain-openai 1.3.x)
# InMemorySaver は langgraph 同梱、LangSmith トレースも追加パッケージ不要
!pip install -U langchain langchain-openai

### 0-2. OpenAI API キーのセットアップ (Colab シークレット方式)

OpenAI API キーは**コードに直接書かず**、Colab の **[シークレット]** 機能で管理します。

**操作手順** (未登録の場合):
1. 画面左のサイドバーにある **鍵アイコン 🔑 [シークレット]** をクリック
2. **[新しいシークレットを追加]** を押す
3. 名前に `OPENAI_API_KEY`、値にあなたの API キーを入力
4. このノートブックからのアクセスを **オン** にする

次のセルは、Colab シークレットからキーを読み取り、環境変数 `OPENAI_API_KEY` に設定します。
LangChain はこの環境変数を自動的に読みます。Colab 以外の環境 (ローカル等) では、
あらかじめ環境変数 `OPENAI_API_KEY` を設定しておけば動きます。

In [ ]:
import os

# Colab のシークレットから OpenAI API キーを読み込み、環境変数に設定する
# Colab 以外の環境では except 側に入り、既存の環境変数 OPENAI_API_KEY をそのまま使う
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab シークレットから OPENAI_API_KEY を読み込みました。")
except ImportError:
    # Colab 以外では環境変数 OPENAI_API_KEY が設定済みとみなす
    print("Colab 以外の環境です。環境変数 OPENAI_API_KEY を使用します。")

# モデル名は変数に集約 ("provider:model" 形式)。研修実施時に最新へ差し替え
MODEL = "openai:gpt-5.4"

print("OpenAI APIキー設定済み:", bool(os.environ.get("OPENAI_API_KEY")))
print("使用モデル:", MODEL)

### 0-3. LangSmith のセットアップ (本章で新規に必要)

本章の後半 (4-4) では **LangSmith** でトレースを可視化します。そのために、LangSmith の
無料アカウントと API キーが必要です。**まだ用意していない場合は、次の手順で準備してください**
(所要 2〜3 分、クレジットカード不要)。

**操作手順**:
1. ブラウザで [smith.langchain.com](https://smith.langchain.com) を開き、**無料アカウントを作成**してログインする
2. 左下の **Settings**(設定) → **API Keys** を開く
3. **[Create API Key]** を押して API キーを発行し、表示された文字列を**コピー**する
   (キーは発行時にしか表示されません)
4. Colab に戻り、左サイドバーの **🔑 [シークレット]** で **[新しいシークレットを追加]**
5. 名前に **`LANGSMITH_API_KEY`**、値にコピーしたキーを貼り付け、**アクセスをオン**にする

> **トレースの有効化に必要なのは「環境変数 2 つだけ」**です
> (`LANGSMITH_TRACING` と `LANGSMITH_API_KEY`)。
> エージェントのコードには一切手を入れません。`create_agent` で作ったエージェントは
> 標準でトレース送信に対応しており、環境変数を見て**自動でトレースを送る**仕掛けが組み込まれています。

次のセルで、トレース送信を有効化します。`LANGSMITH_PROJECT` でトレースの送信先プロジェクトを
分けておくと、後で目的のトレースを見つけやすくなります。

In [ ]:
# --- LangSmith トレースの有効化 (コード変更ゼロ。環境変数を設定するだけ) ---
os.environ["LANGSMITH_TRACING"] = "true"                      # トレース送信を有効化
os.environ["LANGSMITH_PROJECT"] = "langchain-training-day1"   # 送信先プロジェクト名 (整理用)

# Colab シークレットから LangSmith API キーを読み込む
# 非 Colab では環境変数 LANGSMITH_API_KEY を設定済みとみなす
try:
    from google.colab import userdata
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except Exception:
    pass  # 非 Colab では事前に環境変数 LANGSMITH_API_KEY を設定しておく

print("LANGSMITH_TRACING :", os.environ.get("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT :", os.environ.get("LANGSMITH_PROJECT"))
print("LANGSMITH_API_KEY 設定済み:", bool(os.environ.get("LANGSMITH_API_KEY")))

> **トレースをまだ準備できない場合**: `LANGSMITH_API_KEY` が未設定でも、この後の
> checkpointer の実験 (4-2 部分) はそのまま動きます。トレース部分 (4-4) だけ後で実施すれば OK です。
> その場合でも `LANGSMITH_TRACING=true` のままで害はありません (キーがなければ送信されないだけです)。

---

## 1. 天気ツールを用意する (第3章のおさらい)

まず、第3章のハンズオンで使った**天気ツール** `get_weather` をそのまま用意します。
`@tool` で定義し、`city` を受け取って固定の天気を返すダミーツールです。
ここは「動かすだけ」で、本章の新要素ではありません。

In [ ]:
from langchain.tools import tool


@tool
def get_weather(city: str) -> str:
    """指定した都市の現在の天気を取得する。天気・気温の問い合わせにはこのツールを使う。

    Args:
        city: 天気を知りたい都市名 (例: 東京, 大阪)
    """
    # デモ用のダミー実装 (本来は天気 API を呼ぶ想定)
    return f"{city}の天気: 晴れ、気温 24 度"


# 動作確認
print(get_weather.invoke({"city": "東京"}))

---

## 2. checkpointer なしのエージェントは「忘れる」(対比のための出発点)

本題に入る前に、**checkpointer を付けていない**エージェントが会話を忘れることを、もう一度確認します。
第3章で体験した「さっきの都市を覚えていない」現象の再確認です。

第3章と同じく、`create_agent` に `checkpointer` を渡さずにエージェントを作り、
**続けて 2 回 invoke** します。1 回目で「私の名前は Bob です」と伝え、2 回目で「私の名前は?」と聞きます。

**期待される結果**: 2 回目の invoke は、まっさらな State から始まるため、エージェントは
名前を覚えていません (「お名前を伺っていません」といった応答になります)。
`invoke` をまたいだ記憶がどこにも保存されていないからです。

In [ ]:
from langchain.agents import create_agent

# checkpointer なし (第3章と同じ構成)
agent_no_memory = create_agent(
    model=MODEL,
    tools=[get_weather],
    system_prompt="あなたは親切なアシスタントです。",
)

# 1 回目: 名前を伝える
r1 = agent_no_memory.invoke(
    {"messages": [{"role": "user", "content": "こんにちは。私の名前は Bob です。"}]}
)
print("【1回目】", r1["messages"][-1].text)

# 2 回目: 名前を聞く (別の invoke = まっさらな State から始まる)
r2 = agent_no_memory.invoke(
    {"messages": [{"role": "user", "content": "私の名前は?"}]}
)
print("【2回目】", r2["messages"][-1].text)   # => 名前を覚えていない

---

## 3. checkpointer を追加する — 追加は実質 2 行 (4-2)

いよいよ本章の核心です。先ほどのエージェントに **checkpointer** を組み込んで、会話を記憶させます。
開発・検証用の checkpointer として、LangGraph はプロセス内メモリに状態を保存する
`InMemorySaver` を提供しています。

第3章のコードとの差分は、**import を除けば実質 2 行**です。

- **追加①**: `from langgraph.checkpoint.memory import InMemorySaver`
- **追加②**: `create_agent(..., checkpointer=InMemorySaver())`

これだけで、エージェントは実行ステップごとに state のスナップショット (checkpoint) を保存するようになります。
`model` / `tools` / `system_prompt` に続く、`create_agent` の新しい部品が `checkpointer` です。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  # 追加①: import

# 追加②: checkpointer に InMemorySaver を渡すだけ
agent = create_agent(
    model=MODEL,
    tools=[get_weather],
    system_prompt="あなたは親切なアシスタントです。",
    checkpointer=InMemorySaver(),   # ← これで会話を記憶できる (実質ここ 1 行が主役)
)

print("checkpointer 付きエージェントを構成しました。")

### 同じ thread_id で会話が「継続」する

呼び出し側では、`invoke` の第 2 引数に **config** を渡し、その中の `configurable` キーで
**`thread_id`** を指定します。`thread_id` は「**会話の鍵**」です。同じ鍵で invoke すれば、
checkpointer が前回の state を復元してから実行するため、会話が継続します。

先ほどと同じ 2 往復 (名乗る → 名前を聞く) を、今度は**同じ `thread_id="1"`** で実行します。

**期待される結果**: 2 回目の invoke では、checkpointer が thread `"1"` の state を復元し、
モデルには 1 回目の会話を含むメッセージ配列が渡ります。だから今度は名前を答えられます。

In [ ]:
config = {"configurable": {"thread_id": "1"}}   # 会話の鍵

# 1 回目: 名前を伝える (同じ config を渡す)
agent.invoke(
    {"messages": [{"role": "user", "content": "こんにちは。私の名前は Bob です。"}]},
    config,
)

# 2 回目: 同じ thread_id で名前を聞く
result = agent.invoke(
    {"messages": [{"role": "user", "content": "私の名前は?"}]},
    config,
)
print("【thread 1 / 2回目】", result["messages"][-1].text)   # => 「Bob さんですね」など

### thread_id を変えると記憶が「分離」される

次は「分離」を確認します。`thread_id` を **`"2"`** に変えて同じ質問をすると、どうなるでしょうか。
thread `"2"` には Bob の名乗りが存在しないので、エージェントは名前を知りません。

**期待される結果**: 別の鍵 (`"2"`) は別の会話なので、名前を覚えていません。

In [ ]:
config2 = {"configurable": {"thread_id": "2"}}   # 別の鍵 = 別の会話

result_t2 = agent.invoke(
    {"messages": [{"role": "user", "content": "私の名前は?"}]},
    config2,
)
print("【thread 2】", result_t2["messages"][-1].text)   # => 名前を知らない (白紙のスレッド)

### 元の thread_id に戻すと記憶が「残っている」

最後に、`thread_id` を **`"1"`** に戻します。thread `"2"` を使ったことで thread `"1"` の checkpoint が
消えたり上書きされたりはしません。checkpoint は thread ごとに独立して保存されているからです。

**期待される結果**: thread `"1"` に戻すと、Bob の名前を**まだ覚えています**。
「同じ鍵なら続き、違う鍵なら新規、元の鍵に戻れば元の続き」——これが checkpointer の動作モデルです。

In [ ]:
# thread_id="1" に戻す (config を再利用)
result_back = agent.invoke(
    {"messages": [{"role": "user", "content": "もう一度聞きます。私の名前は?"}]},
    config,
)
print("【thread 1 に復帰】", result_back["messages"][-1].text)   # => まだ Bob を覚えている

---

## 4. messages の件数が累積していくのを観察する

もう 1 つ、ぜひ観察してほしいのが `result["messages"]` の**件数**です。
checkpointer 付きのエージェントでは、`invoke` のたびに戻り値の messages が**累積**していきます。
これは「state に履歴が積まれ、毎回その全体がモデルに送られている」ことの直接の証拠です。

新しい thread `"3"` を使い、3 回 invoke して、そのたびに messages 件数がどう増えるかを見ます。

**期待される結果**: invoke のたびに messages の件数が増えていきます
(ツール呼び出しを挟むと、その分さらに増えます)。会話を重ねるほどモデルへの入力は太っていく——
この観察が、本章 4-3 で扱う「長い会話の問題」の伏線になります。

In [ ]:
config3 = {"configurable": {"thread_id": "3"}}

turns = [
    "東京の天気を教えて。",
    "ありがとう。ところで私は旅行が好きです。",
    "私の趣味は何だっけ?",
]

for i, text in enumerate(turns, start=1):
    res = agent.invoke({"messages": [{"role": "user", "content": text}]}, config3)
    print(f"invoke {i} 回目 | messages 件数 = {len(res['messages']):2d} | 最終応答: {res['messages'][-1].text[:40]}")

> **補足: config なしで invoke するとどうなる?**
> checkpointer を設定したエージェントを **config なし (= `thread_id` なし)** で `invoke` すると
> **エラー**になります。checkpointer は `thread_id` を主キーとして checkpoint を保存・復元するため、
> 鍵を渡されなければ「どの会話か」を特定できないのです。
> エラーメッセージに驚かず、「鍵 (thread_id) を忘れた」と読み替えてください。
> (下のセルはあえてエラーを起こして確認する例です。`try/except` で受けています。)

In [ ]:
# あえて config なしで invoke してエラーを確認する (checkpointer 付きエージェント)
try:
    agent.invoke({"messages": [{"role": "user", "content": "私の名前は?"}]})
    print("エラーになりませんでした (環境によっては挙動が異なる場合があります)")
except Exception as e:
    print("想定どおりエラーになりました (thread_id が必要):")
    print(" ", type(e).__name__, "-", str(e)[:120])

---

## 5. LangSmith でトレースを有効化する — コード変更ゼロ (4-4)

ここからは本章の 2 つ目のテーマ、**可観測性**です。これまで私たちが観察できていたのは
`result["messages"]` を print した結果だけでした。**LangSmith** を使うと、エージェントの全実行——
ループの周回数・ツールの引数・トークン消費——を、ブラウザで可視化できます。

そして重要なのは、トレースを有効にするための**コード変更はゼロ**だということです。
必要な準備は **0-3 ですでに完了**しています (環境変数 `LANGSMITH_TRACING` と `LANGSMITH_API_KEY` の設定)。
あとは**今まで通りエージェントを動かすだけ**で、`invoke` のたびにトレースが LangSmith へ送信されます。

下のセルを実行したら、ブラウザで [smith.langchain.com](https://smith.langchain.com) を開き、
プロジェクト **`langchain-training-day1`** を選ぶと、いま実行したトレースが現れます。

In [ ]:
# トレースを送るために、ふつうに invoke するだけ (コードに特別な記述は不要)
config_trace = {"configurable": {"thread_id": "trace-demo"}}

res = agent.invoke(
    {"messages": [{"role": "user", "content": "東京の天気は?"}]},
    config_trace,
)
print("応答:", res["messages"][-1].text)
print()
print("→ smith.langchain.com を開き、プロジェクト 'langchain-training-day1' でこのトレースを確認してください。")

### config に tags / metadata を付けてトレースを整理する

トレースは、`config` に **`tags`** と **`metadata`** を渡すと、付加情報を記録できて、
LangSmith 上で検索・フィルタできるようになります。`config` はすでに `thread_id` で使っている
あの辞書です。注意点は、**`tags` や `metadata` は `configurable` の中ではなく、同じ階層**に書くことです。

```python
config = {
    "configurable": {"thread_id": "1"},   # 会話の鍵 (4-2)
    "tags": ["day1", "hands-on"],         # ← configurable と同じ階層
    "metadata": {"user_id": "trainee_01"},# ← configurable と同じ階層
}
```

**期待される結果**: 下のセルを実行すると、LangSmith のトレースに `day1` / `hands-on` のタグと
`user_id: trainee_01` のメタデータが付きます。トレース一覧でタグによる絞り込みができることを
確認してみてください。

In [ ]:
config_tagged = {
    "configurable": {"thread_id": "trace-demo"},   # 会話の鍵
    "tags": ["day1", "hands-on"],                  # 検索用のラベル (configurable と同じ階層)
    "metadata": {"user_id": "trainee_01"},         # 任意のキーバリュー (同じ階層)
}

res = agent.invoke(
    {"messages": [{"role": "user", "content": "大阪の天気は?"}]},
    config_tagged,
)
print("応答:", res["messages"][-1].text)
print("→ LangSmith でこのトレースに tags / metadata が付いていることを確認してください。")

---

## 6. あえてツールを 2 回呼ばせて、ループが増える様子を見る

トレースの読解に入る前に、**ループ周回数が増える**例を 1 つ作っておきます。
「**東京と大阪の天気を比べて**」と聞くと、モデルは `get_weather` を**2 回**(東京・大阪) 呼ぶ必要があります。
1 ツールで完結する質問よりも、エージェントループが多く回ります。

**期待される結果**: 軌跡に `get_weather` の呼び出しが**2 回**現れます。
LangSmith のトレースでも、`model → tools → model → tools → model` のように周回が増えて見えるはずです
(次のセクションのワークシートで確認します)。

In [ ]:
config_compare = {
    "configurable": {"thread_id": "compare-demo"},
    "tags": ["day1", "hands-on", "multi-tool"],
}

res = agent.invoke(
    {"messages": [{"role": "user", "content": "東京と大阪の天気を比べてどちらが過ごしやすい?"}]},
    config_compare,
)

# 軌跡から、呼ばれたツール名を順番に取り出す
called = []
for m in res["messages"]:
    for tc in (getattr(m, "tool_calls", None) or []):
        called.append(tc["name"])

print("呼ばれたツール (順番):", called)            # get_weather が 2 回現れるはず
print("messages 件数        :", len(res["messages"]))
print("最終応答             :", res["messages"][-1].text)

---

## 7. トレース読解ワークシート — 3 点チェック

ここがハンズオン 4-A の総仕上げです。ブラウザで [smith.langchain.com](https://smith.langchain.com) を開き、
プロジェクト `langchain-training-day1` から**トレースを 1 つ開いて**、下の **3 点チェック**を埋めてください。
1 回の `invoke` が 1 つのルート **run** として表示され、その配下にモデル呼び出しやツール実行が
**階層構造**でぶら下がっています。この木構造は、第3章で学んだエージェントループそのものです。

### 読み方の型 — 3 点チェック

1. **① ループは何周したか** — ルート run の配下の「モデル呼び出し → ツール実行」の繰り返しを数えます。
   `model → tools → model` で終われば 1 周、ツールを 2 回呼んでいれば 2 周です。
2. **② どのツールが、どの引数で呼ばれたか** — 各 run をクリックすると入出力が見えます。
   ツール run なら、モデルが生成した引数と、ツールが返した結果が確認できます。
3. **③ トークンをどこで消費したか** — run ごとに入出力トークン数とレイテンシが表示されます。

---

### ワークシート (記入してください)

まず **セクション 6 の「東京と大阪を比べて」のトレース**を開いて記入します。

| チェック項目 | 記入欄 |
|---|---|
| ① ループは何周したか (model→tools の繰り返し回数) | （例: 2 周） |
| ② 呼ばれたツール名と引数 (1 回目) | （例: get_weather, city=東京） |
| ② 呼ばれたツール名と引数 (2 回目) | （例: get_weather, city=大阪） |
| ③ 合計トークン (Total Tokens) | （トレース画面の値を記入） |
| ③ いちばんトークンを使った run はどれか | （例: 最後のモデル呼び出し） |

---

### checkpointer の効果をトレースで確認する

次に、**セクション 3 で実行した thread `"1"` のトレース** (「私の名前は?」と聞いた 2 回目の invoke) を開きます。
**モデル呼び出しの入力 (Input)** を見てください。

| 確認項目 | 記入欄 |
|---|---|
| 2 回目の invoke のモデル入力に、1 回目の「私の名前は Bob です」が含まれているか | （含まれている / いない） |
| 含まれているとしたら、それは何の働きによるものか | （例: checkpointer が state を復元したため） |

> **ここが本章前半と後半のつながり**です。「checkpointer が state を復元し、履歴全体がモデルに送られる」という
> 4-2 の説明を、トレースは**物的証拠**として見せてくれます。print では追いにくかったこの事実が、
> ブラウザで一目で確認できます。

---

## まとめ — ハンズオン 4-A で確認したこと

| 節 | 確認したこと | キーポイント |
|---|---|---|
| 4-2 | checkpointer 追加 | `InMemorySaver` を import し `create_agent(..., checkpointer=...)`。**実質 2 行**で会話を記憶 |
| 4-2 | thread_id で継続/分離 | `config={"configurable": {"thread_id": ...}}`。同じ鍵=継続、違う鍵=分離、元の鍵=元の続き |
| 4-2 | messages の累積 | invoke のたびに messages が積み上がる (= 履歴全体が毎回モデルに送られている証拠) |
| 4-4 | LangSmith 有効化 | 環境変数 2 つ + プロジェクトで**コード変更ゼロ**。`config` の `tags` / `metadata` で整理 |
| 4-4 | 3 点チェック | ①ループ周回数 ②ツールと引数 ③トークン消費。checkpointer の効果も入力で確認できる |

第3章で未解決だった「会話を覚えない」問題が、checkpointer の 2 行で解決しました。
そして「エージェントの頭の中が見えない」問題も、環境変数 2 つで解決しました。

### 明日からの合言葉

**「エージェントを動かしたら、必ずトレースを見る」**。第5章で MCP サーバーからツールを調達するとき、
第6章で Middleware の挙動を確かめるとき、デバッグの主戦場はすべてこのトレース画面です。

### 次は演習 4-B へ

このハンズオンで習得した部品 (checkpointer / thread_id / tags / metadata / トレース読解) を、
演習 4-B では「**社内 IT ヘルプデスクエージェント v2**」に応用します (ヘルプデスク Step 3)。
問い合わせのたびに会話を忘れる v1 を、**社員ごとに会話を記憶し、運用チームがトレースで診断できる v2** へ
拡張します。`# TODO` を埋める形式です。